In [ ]:
!pip install -q opencv-python

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files


def load_image():
    uploaded = files.upload()
    name = next(iter(uploaded))
    data = np.frombuffer(uploaded[name], np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def show(items, cols=2, figsize=(12, 10)):
    rows = (len(items) + cols - 1) // cols
    plt.figure(figsize=figsize)
    for i, (title, img) in enumerate(items, 1):
        plt.subplot(rows, cols, i)
        if img.ndim == 2:
            plt.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
img = load_image()

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 11, 2)

show([('Исходное изображение', img),
      ('Оттенки серого', gray),
      ('threshold()', binary),
      ('adaptiveThreshold()', adaptive)])

In [ ]:
dark = load_image()
gray_dark = cv2.cvtColor(dark, cv2.COLOR_BGR2GRAY)
equalized = cv2.equalizeHist(gray_dark)

show([('Исходное (затемнённое)', gray_dark),
      ('После equalizeHist()', equalized)], cols=2, figsize=(12, 5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(gray_dark.ravel(), bins=256, range=(0, 256))
axes[0].set_title('Гистограмма исходного изображения')
axes[1].hist(equalized.ravel(), bins=256, range=(0, 256))
axes[1].set_title('Гистограмма выровненного изображения')
for ax in axes:
    ax.set_xlabel('Яркость')
    ax.set_ylabel('Число пикселей')
plt.tight_layout()
plt.show()

In [ ]:
noisy = load_image()

blurred = cv2.blur(noisy, (5, 5))
gaussian = cv2.GaussianBlur(noisy, (5, 5), 0)
median = cv2.medianBlur(noisy, 5)

show([('Исходное (с шумами)', noisy),
      ('blur()', blurred),
      ('GaussianBlur()', gaussian),
      ('medianBlur()', median)])

In [ ]:
objects = load_image()
gray_obj = cv2.cvtColor(objects, cv2.COLOR_BGR2GRAY)

blur_obj = cv2.GaussianBlur(gray_obj, (5, 5), 0)
k = max(31, (min(gray_obj.shape) // 3) | 1)
background = cv2.medianBlur(gray_obj, k)
diff = cv2.absdiff(blur_obj, background)
_, bin_obj = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

kernel = np.ones((5, 5), np.uint8)
eroded = cv2.erode(bin_obj, kernel, iterations=1)
dilated = cv2.dilate(bin_obj, kernel, iterations=1)

show([('Исходное изображение', objects),
      ('Бинарное изображение', bin_obj),
      ('erode()', eroded),
      ('dilate()', dilated)])